# title: "SIMPlex Prostate Cancer — integrative single-nucleus and spatial analysis"
subtitle: "Manuscript figures: Extended Data Fig. 3"
author: "Marcos Tito Machado"
output: html_document



In [ ]:
# DATA_ROOT, HDF5 library, palettes — single source of truth for paths.
source(here::here("config.R"))

In [ ]:
# HDF5 library is loaded by config.R
library(semla)
library(tibble)
library(Seurat)
options(Seurat.object.assay.version = "v5")
library(patchwork)
library(plotly)
library(singlet)
library(RcppML)
library(ggplot2)
library(dplyr)
library(tidyr)
library(viridis)
library(pheatmap)
library(cowplot)
library(corrplot)
library(RColorBrewer)
library(heatmap3)
library(harmony)



### Settings 


In [ ]:
rootObj <- paste0(SN_RDS, "/prostate_cancer/integrated/cafs/")
wd <- paste0(FIGS_ROOT, "/prostate_cancer/cafs/")
if (!dir.exists(rootObj)) {
    dir.create(rootObj, recursive = TRUE)
}
if (!dir.exists(wd)) {
    dir.create(wd, recursive = TRUE)
}



### Merge datasets


In [ ]:
spl_paths <- list(
    "pt10" = list(
        path = paste0(SN_RDS, "/prostate_cancer/pt10_HD_seuratAnnotation.rds"),
        subtype = "NA"
    ),
    "pt20" = list(
        path = paste0(SN_RDS, "/prostate_cancer/pt20_HD_seuratAnnotation.rds"),
        subtype = "NA"
    )
)
spls <- list()
for (splname in names(spl_paths)) {
    spl <- readRDS(spl_paths[[splname]]$path)
    spl@meta.data$sample <- splname
    spl@meta.data$subtype <- spl_paths[[splname]]$subtype
    spls[[splname]] <- spl
}
merged <- merge(x = spls[[1]], y = spls[-1])
table(merged@meta.data$sample)



### Integrate datasets

#### Integrate all 


In [ ]:
wd <- paste0(FIGS_ROOT, "/prostate_cancer/total_integration/")
DefaultAssay(merged) <- "RNA"
set.seed(1)
merged <- merged %>%
  NormalizeData() %>%
  FindVariableFeatures() %>%
  ScaleData() %>%
  RunPCA() %>%
  RunUMAP(dims = 1:30)
# plot pre integration
ggsave(
    filename = paste0(wd, "pca_pre_integration.pdf"),
    plot = DimPlot(merged, reduction = "pca", group.by = "sample"),
    width = 10,
    height = 10
)
ggsave(
    filename = paste0(wd, "umap_pre_integration.pdf"),
    plot = DimPlot(merged, reduction = "umap", group.by = c("sample", "seuratAnnotation_major", "subtype"), ncol = 3),
    width = 30,
    height = 10
)
# integrate
int.all <- merged %>% 
    RunHarmony("sample", plot_convergence = TRUE, theta = 0) # min thetha since we expect a lot of biological variation
int.all <- RunUMAP(int.all, dims = 1:30, reduction = "harmony")

# plot post integration
ggsave(
    filename = paste0(wd, "pca_post_integration.pdf"),
    plot = DimPlot(int.all, reduction = "harmony", group.by = "sample"),
    width = 10,
    height = 10
)
ggsave(
    filename = paste0(wd, "umap_post_integration.pdf"),
    plot = DimPlot(int.all, reduction = "umap", group.by = c("sample", "seuratAnnotation_major", "subtype"), ncol = 3),
    width = 30,
    height = 10
)
wd <- paste0(FIGS_ROOT, "/prostate_cancer/cafs/")



## Integrate CAFs only 


In [ ]:
merged.cafs <- subset(merged, subset = seuratAnnotation_major == "CAFs")

In [ ]:
set.seed(1)
merged.cafs <- merged.cafs %>%
  NormalizeData() %>%
  FindVariableFeatures() %>%
  ScaleData() %>%
  RunPCA() %>%
  RunUMAP(dims = 1:30)
# plot pre integration
ggsave(
    filename = paste0(wd, "pca_pre_integration.pdf"),
    plot = DimPlot(merged.cafs, reduction = "pca", group.by = "sample"),
    width = 10,
    height = 10
)
ggsave(
    filename = paste0(wd, "umap_pre_integration.pdf"),
    plot = DimPlot(merged.cafs, reduction = "umap", group.by = c("sample", "seuratAnnotation_major", "subtype"), ncol = 3),
    width = 30,
    height = 7
)
# integrate
int <- merged.cafs %>% 
    RunHarmony("sample", plot_convergence = TRUE, theta = 0) # min theta since we want to preserve a lot of biological variation
int <- RunUMAP(int, dims = 1:30, reduction = "harmony") 

# plot post integration
ggsave(
    filename = paste0(wd, "pca_post_integration.pdf"),
    plot = DimPlot(int, reduction = "harmony", group.by = "sample"),
    width = 10,
    height = 10
)
ggsave(
    filename = paste0(wd, "umap_post_integration.pdf"),
    plot = DimPlot(int, reduction = "umap", group.by = c("sample", "seuratAnnotation_major", "subtype"), ncol = 3),
    width = 30,
    height = 7
)



### Integrated clustering of CAFS 


In [ ]:
set.seed(1)
int <- FindNeighbors(int, dims = 1:30)
for (res in c(0.1, 0.25, .5, 0.8, 1, 1.25)){
  int <- FindClusters(int, resolution = res, algorithm = 1, verbose = FALSE)
}

d1 <- DimPlot(int, reduction = "umap", group.by = "RNA_snn_res.0.1", label = TRUE) + 
  DimPlot(int, reduction = "umap", group.by = "RNA_snn_res.0.25", label = TRUE) + 
  DimPlot(int, reduction = "umap", group.by = "RNA_snn_res.0.5", label = TRUE)
d2 <- DimPlot(int, reduction = "umap", group.by = "RNA_snn_res.0.8", label = TRUE) + 
  DimPlot(int, reduction = "umap", group.by = "RNA_snn_res.1", label = TRUE) + 
  DimPlot(int, reduction = "umap", group.by = "RNA_snn_res.1.25", label = TRUE)

pdf(paste0(wd, "umap_integratedClusters.pdf"), width = 20, height = 10)
d1 / d2
dev.off()



### If REGENERATING PLOTS with renamed clusters


In [ ]:
renamed = TRUE
if (renamed){
    wd <- paste0(FIGS_ROOT, "/prostate_cancer/cafs/RENAMED_")
}

In [ ]:
if (renamed == FALSE){
    int@meta.data$seurat_clusters <- int@meta.data$RNA_snn_res.0.5
}
Idents(int) <- "seurat_clusters"
ggsave(
    filename = paste0(wd, "umap_post_integration_clusters.pdf"),
    plot = DimPlot(int, reduction = "umap", group.by = c("sample", "subtype", "seurat_clusters"), ncol = 2, label = TRUE, raster = FALSE),
    width = 16,
    height = 12
)



## Comparing our CAF clusters to literature

The goal on this section is to compare the subpopulations in https://www.nature.com/articles/s41467-023-39762-1 
and compare it with our CAF cell states.



In [ ]:
mCAF <- c("MMP11", "COMP", "POSTN", "COL1A1", "COL1A2", "COL10A1", "COL11A1", "COL8A1", "COL1A2", "COL12A1", "COL3A1", "COL8A1", "COL5A2")
iCAF <- c("CFD", "PLA2G2A", "APOD", "CXCL14", "CD34", "CCLX12", "IL6")
vCAF <- c("NOTCH3", "COL18A1", "MCAM", "MHY11", "RERGL", "ACTA2")
tCAF <- c("PDPN", "MME", "TMEM158", "NDRG1")
ifnCAF <- c("IL32", "CXCL9", "CXCL10", "CXCL11", "IDO1")
apCAF <- c("HLA-DRA", "HLA-DRB1", "CD74")
rCAF <- c("CCL21", "CCL19", "IGFBP5")

caf_celltypes <- list(
  mCAF = mCAF,
  iCAF = iCAF,
  vCAF = vCAF,
  tCAF = tCAF,
  ifnCAF = ifnCAF,
  apCAF = apCAF,
  rCAF = rCAF
)

nat_all_genes <- unique(c(mCAF, iCAF, vCAF, tCAF, ifnCAF, apCAF, rCAF))

Idents(int) <- "seurat_clusters"
p <- DotPlot(int, features = nat_all_genes) + 
        coord_flip() +
        scale_color_gradientn(colors = c("blue", "white", "darkred")) +
        theme(axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1))
ggsave(paste0(wd, "dotplot_nat_markers.pdf"), p, width = 10, height = 10)

ggsave(
    filename = paste0(wd, "umap_nat_markers.pdf"),
    plot = FeaturePlot(int, features = nat_all_genes),
    width = 50,
    height = 70,
    limitsize = FALSE
)



#### Gene expression similarity heatmap 



In [ ]:
nat <- readRDS(paste0(EXT_REFS, "/CAF_Cords2023/BREAST_fibro_tumour.rds"))
Idents(int) <- "seurat_clusters"
Idents(nat) <- "CAFtype"
caf_cells <- rownames(int@meta.data)
int <- FindVariableFeatures(int, selection.method = "vst", nfeatures = 3000)
int <- ScaleData(int)

genesNat <- VariableFeatures(nat)
genesCafs <- VariableFeatures(int)

natAvg <- AverageExpression(nat, group.by = "CAFtype", return.seurat = TRUE, assays = "RNA")
natAvg <- ScaleData(natAvg)
natData <- FetchData(object = natAvg, vars = genesNat, layer = "scale.data")
cafsAvg <- AverageExpression(int, group.by = "seurat_clusters", return.seurat = TRUE, assays = "RNA")
cafsAvg <- ScaleData(cafsAvg)
cafsData <- FetchData(object = cafsAvg, vars = genesCafs, layer = "scale.data")

# Extract gene names from each dataset, find common genes, merge datasets
genes_nat <- colnames(natData)
genes_cafs <- colnames(cafsData)
common_genes <- Reduce(intersect, list(genes_nat, genes_cafs))
nat_common <- natData[, common_genes]
# rownames(nat_common) <- paste0("Spatial_", rownames(spatial_common))
cafs_common <- cafsData[, common_genes]
# rownames(simplex_common) <- paste0("SIMPlex_", rownames(simplex_common))


# Calculate correlation between spatial and combined dataset
correlation <- cor(t(nat_common), t(cafs_common))

# Save the plot generated by pheatmap
pdf(paste0(wd, "2023_Cords_natComms_COMPARISON.pdf"), width = 10, height = 10)
heatmap3(correlation, margins = c(10, 10), scale = "none")
dev.off()

print(paste0("Based on ", length(common_genes), " common genes"))



## GSEA CAF subpopulations


In [ ]:
library(singleseqgset)
library(msigdbr)

#### Select Gene Set of interest
# H: hallmark gene sets
# C1: positional gene sets
# C2: curated gene sets

category <- "H" # category from Human MSigDB Collections
h.human <- msigdbr(species="Homo sapiens",category=category)
subcategory <- ""
h.human <- h.human %>% filter(gs_subcat == subcategory)
h.names <- unique(h.human$gs_name)
h.sets <- vector("list",length=length(h.names))
names(h.sets) <- h.names
for (i in names(h.sets)) {
    h.sets[[i]] <- pull(h.human[h.human$gs_name==i,"gene_symbol"])
}
#### Filter out non relevant pathways
# Define the list of pathways to remove
remove_pathways <- c(
  "HALLMARK_BILE_ACID_METABOLISM",         # Liver-specific  
  "HALLMARK_PANCREAS_BETA_CELLS",          # Pancreatic function  
  "HALLMARK_SPERMATOGENESIS",              # Testis-specific  
  "HALLMARK_MYOGENESIS",                   # Muscle development  
  "HALLMARK_ADIPOGENESIS",                 # Fat cell differentiation  
  "HALLMARK_FATTY_ACID_METABOLISM",        # General lipid metabolism  
  "HALLMARK_HEME_METABOLISM",              # Blood-related, not CAF-specific  
  "HALLMARK_CHOLESTEROL_HOMEOSTASIS",      # Unrelated lipid metabolism  
  "HALLMARK_PEROXISOME",                   # General cell detox pathway  
  "HALLMARK_XENOBIOTIC_METABOLISM",        # Detox, not breast cancer-related  
  "HALLMARK_UV_RESPONSE_DN",               # Skin cancer-related  
  "HALLMARK_UV_RESPONSE_UP"                # Skin cancer-related  
)
# Filter the list, keeping only relevant pathways
h.sets <- h.sets[!names(h.sets) %in% remove_pathways]
##### Change names (aesthetic purposes only)
names(h.sets) <- gsub("HALLMARK_", "", names(h.sets))
#### Log Fold change
Idents(int) <- "seurat_clusters"
int$seurat_clusters <- Idents(int)
mtx <- GetAssayData(int, assay = "RNA", layer = "data")
logfc.cafs <- logFC(cluster.ids = int@meta.data$seurat_clusters, expr.mat = mtx)
names(logfc.cafs)
#### Run GSEA
gse.res <- wmw_gsea(expr.mat=mtx, cluster.cells=logfc.cafs[[1]],
                    log.fc.cluster=logfc.cafs[[2]],
                    gene.sets=h.sets)
res.stats <- gse.res[["GSEA_statistics"]]
res.pvals <- gse.res[["GSEA_p_values"]]
res.pvals <- apply(res.pvals,2,p.adjust,method="fdr") #Correct for multiple comparisons
res.stats[order(res.stats[,1],decreasing=TRUE)[1:10],] #Top gene sets enriched by z scores
#Plot the z scores with heatmap3
pdf(paste0(wd, "GSEA.pdf"), width = 12, height = 12)
heatmap3(res.stats, margins = c(7, 34))
dev.off()



#### add subclusters back to main object


In [ ]:
if (renamed == TRUE){
    int$renamed_clusters <- paste0("", int$seurat_clusters)
} else {
    int$renamed_clusters <- paste0("CAF_", int$seurat_clusters)
}
renamed_clusters <- int$renamed_clusters
names(renamed_clusters) <- Cells(int)

int.all$CAF_clusters_integrated <- NA
int.all$CAF_clusters_integrated[names(renamed_clusters)] <- renamed_clusters

# Create a new metadata column for CAF subtypes
int.all$seuratAnnotation_major_cafsInt <- as.character(int.all$seuratAnnotation_major)
int.all$seuratAnnotation_major_cafsInt[int.all$seuratAnnotation_major == "CAFs"] <- int.all$CAF_clusters_integrated[int.all$seuratAnnotation_major == "CAFs"]
int.all$seuratAnnotation_major_cafsInt <- factor(int.all$seuratAnnotation_major_cafsInt)

ggsave(
    filename = paste0(wd, "CAFs_all_integratedUMAP.pdf"),
    plot = DimPlot(int.all, reduction = "umap", group.by = c("seuratAnnotation_major_cafsInt"), ncol = 1, label = TRUE, raster = FALSE),
    width = 10,
    height = 10
)



### Integrated Spatial mapping 

##### load visium HD data and save object
only need to do this once and save the visium HD object 


In [ ]:
library(stringr)
library(arrow)
samples <- unique(int.all$sample)
spls <- list()
for (sample in samples){
st.dir <- paste0(SPACERANGER, "/prostate_cancer/HD/", sample, "_HD/binned_outputs")
res <- paste(0, c(8, 16), "um", sep = "")
all.res <- paste(res, collapse = "|")
res.dir <- list.dirs(st.dir, recursive = FALSE) |> 
  stringr::str_subset(pattern = all.res)
infoTable <- data.frame(
  samples = list.files(res.dir,
                       full.names = TRUE, recursive = TRUE,
                       pattern = paste0("^filtered_feature.+.h5$")
  ),
  spotfiles = list.files(res.dir,
                         full.names = TRUE, recursive = TRUE,
                         pattern = "parquet$|positions.csv$"
  ),
  imgs = list.files(res.dir,
                    recursive = TRUE,
                    full.names = TRUE, pattern = "hires"
  ) |>
    str_subset(all.res),
  json = list.files(st.dir,
                    recursive = TRUE,
                    full.names = TRUE, pattern = "^scalefactors"
  ) |>
    str_subset(all.res),
  resolution = res,
  sample_id = sample
)
se.hd <- ReadVisiumData(infoTable)
saveRDS(se.hd, paste0(SPATIAL_RDS, "/prostate_cancer/", sample, "_visHD.rds"))
}



###### map 


In [ ]:
samples <- unique(int.all$sample)
spls <- list()
for (sample in samples){
    vis <- readRDS(paste0(SPATIAL_RDS, "/prostate_cancer/", sample, "_visHD.rds"))
    vis <- SubsetSTData(vis, expression = resolution == "016um") # use this resolution 

    ## Map w/ semla NNLS
    ident <- "seuratAnnotation_major_cafsInt"
    celltype <- "major_integrated_CAFs"
    DefaultAssay(vis) <- "Spatial"
    DefaultAssay(int.all) <- "RNA"
    vis <- vis |>
    NormalizeData() |>
    FindVariableFeatures(nfeatures = nrow(vis)) |>
    ScaleData()
    int.all <- int.all |>
    FindVariableFeatures(nfeatures = 5000)
    vis <- RunNNLS(vis, singlecell_object = int.all, spatial_assay = DefaultAssay(vis), singlecell_assay = DefaultAssay(int.all), groups = ident, slot = "data", assay_name = celltype)
    DefaultAssay(vis) <- celltype

    spls[[sample]] <- vis
}
mergedVis <- MergeSTData(spls[[1]], spls[-1])
table(mergedVis@meta.data$sample_id)
DefaultAssay(mergedVis) <- celltype

p <- MapFeatures(mergedVis, features = rownames(mergedVis)[grepl("CAF", rownames(mergedVis))], colors = viridis::turbo(11), label_by = "sample_id", pt_size = 0.35, override_plot_dims = TRUE, scale = "shared")
ggsave(paste0(wd, "map/", "all_integratedMap_patient_specific.jpg"), p, width = 36, height = 18, dpi = 300, limitsize = FALSE)
p <- MapFeatures(mergedVis, features = rownames(mergedVis), colors = viridis::turbo(11), label_by = "sample_id", pt_size = 0.4, override_plot_dims = TRUE, scale = "shared")
ggsave(paste0(paste0(FIGS_ROOT, "/prostate_cancer/total_integration/"), "map/","all_integratedMap.jpg"), p, width = 105, height = 20, dpi = 300, limitsize = FALSE)



##### Deconvolution of the patient labels 


In [ ]:
samples <- unique(int.all$sample)
spls <- list()
for (sample in samples){
    vis <- readRDS(paste0(SPATIAL_RDS, "/prostate_cancer/", sample, "_visHD.rds"))
    vis <- SubsetSTData(vis, expression = resolution == "016um") # use this resolution 

    ## Map w/ semla NNLS
    ident <- "sample"
    celltype <- "sample_id_mapping"
    DefaultAssay(vis) <- "Spatial"
    DefaultAssay(int.all) <- "RNA"
    vis <- vis |>
    NormalizeData() |>
    FindVariableFeatures(nfeatures = nrow(vis)) |>
    ScaleData()
    int.all <- int.all |>
    FindVariableFeatures(nfeatures = 5000)
    vis <- RunNNLS(vis, singlecell_object = int.all, spatial_assay = DefaultAssay(vis),
                   singlecell_assay = DefaultAssay(int.all), groups = ident, slot = "data",
                   assay_name = celltype)
    DefaultAssay(vis) <- celltype

    spls[[sample]] <- vis
}
mergedVis <- MergeSTData(spls[[1]], spls[-1])
table(mergedVis@meta.data$sample_id)
DefaultAssay(mergedVis) <- celltype

p <- MapFeatures(mergedVis, features = rownames(mergedVis), colors = viridis::turbo(11), label_by = "sample_id", pt_size = 0.4, override_plot_dims = TRUE, scale = "shared")
ggsave(paste0(paste0(FIGS_ROOT, "/prostate_cancer/total_integration/"), "map/","all_patientID_map.jpg"), p, width = 10, height = 10, dpi = 300, limitsize = FALSE)

In [ ]:
saveRDS(int, paste0(rootObj, "integrated_cafs.rds"))
saveRDS(int.all, paste0(paste0(SN_RDS, "/prostate_cancer/integrated/"), "integrated_all.rds"))



### CAF cell state markers


In [ ]:
Idents(int) <- "seurat_clusters"
caf.markers <- FindAllMarkers(int, assay = "RNA")
caf.markers %>%
    group_by(cluster) %>%
    top_n(n = 4, wt = avg_log2FC) -> top

p <- DotPlot(int, features = top$gene) + 
        coord_flip() +
        scale_color_gradientn(colors = c("blue", "white", "darkred")) +
        theme(axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1))
ggsave(paste0(wd, "markers_dotplot.pdf"), p, width = 8, height = 10)



## Rename clusters to putative populations


In [ ]:
int@meta.data$CAF_clusters_integratedRenamed <- int@meta.data$seurat_clusters
caf.renames <- c(
    "0" = "apCAF_1",
    "1" = "apCAF_2",
    "2" = "mCAF/iCAF",
    "3" = "iCAF_1",
    "4" = "apCAF_3",
    "5" = "pt20_CAF",
    "6" = "unlikelyCAF"
)
int@meta.data$CAF_clusters_integratedRenamed <- plyr::mapvalues(int@meta.data$CAF_clusters_integratedRenamed, from = names(caf.renames), to = caf.renames)
int@meta.data$seurat_clusters <- int@meta.data$CAF_clusters_integratedRenamed



################################################################## Epithelial analysis #####################################################################################################

### Settings 


In [ ]:
rootObj <- paste0(SN_RDS, "/prostate_cancer/integrated/epithelial/")
wd <- paste0(FIGS_ROOT, "/prostate_cancer/epithelial/")
if (!dir.exists(rootObj)) {
    dir.create(rootObj, recursive = TRUE)
}
if (!dir.exists(wd)) {
    dir.create(wd, recursive = TRUE)
}



## Integrate Epithelial only 


In [ ]:
merged.epi <- subset(merged, subset = seuratAnnotation_major == "Epithelial")

In [ ]:
set.seed(1)
merged.epi <- merged.epi %>%
  NormalizeData() %>%
  FindVariableFeatures() %>%
  ScaleData() %>%
  RunPCA() %>%
  RunUMAP(dims = 1:30)
# plot pre integration
ggsave(
    filename = paste0(wd, "pca_pre_integration.pdf"),
    plot = DimPlot(merged.epi, reduction = "pca", group.by = "sample"),
    width = 10,
    height = 10
)
ggsave(
    filename = paste0(wd, "umap_pre_integration.pdf"),
    plot = DimPlot(merged.epi, reduction = "umap", group.by = c("sample", "seuratAnnotation_major", "subtype"), ncol = 3),
    width = 30,
    height = 7
)
# integrate
int <- merged.epi %>% 
    RunHarmony("sample", plot_convergence = TRUE, theta = 0) # min theta since we want to preserve a lot of biological variation
int <- RunUMAP(int, dims = 1:30, reduction = "harmony") 

# plot post integration
ggsave(
    filename = paste0(wd, "pca_post_integration.pdf"),
    plot = DimPlot(int, reduction = "harmony", group.by = "sample"),
    width = 10,
    height = 10
)
ggsave(
    filename = paste0(wd, "umap_post_integration.pdf"),
    plot = DimPlot(int, reduction = "umap", group.by = c("sample", "seuratAnnotation_major", "subtype"), ncol = 3),
    width = 30,
    height = 7
)



### Integrated clustering of Epithelial 


In [ ]:
set.seed(1)
int <- FindNeighbors(int, dims = 1:30)
for (res in c(0.1, 0.25, .5, 0.8, 1, 1.25)){
  int <- FindClusters(int, resolution = res, algorithm = 1, verbose = FALSE)
}

d1 <- DimPlot(int, reduction = "umap", group.by = "RNA_snn_res.0.1", label = TRUE) + 
  DimPlot(int, reduction = "umap", group.by = "RNA_snn_res.0.25", label = TRUE) + 
  DimPlot(int, reduction = "umap", group.by = "RNA_snn_res.0.5", label = TRUE)
d2 <- DimPlot(int, reduction = "umap", group.by = "RNA_snn_res.0.8", label = TRUE) + 
  DimPlot(int, reduction = "umap", group.by = "RNA_snn_res.1", label = TRUE) + 
  DimPlot(int, reduction = "umap", group.by = "RNA_snn_res.1.25", label = TRUE)

pdf(paste0(wd, "umap_integratedClusters.pdf"), width = 20, height = 10)
d1 / d2
dev.off()



### REGENERATING PLOTS with renamed clusters


In [ ]:
renamed = FALSE
if (renamed){
    wd <- paste0(FIGS_ROOT, "/prostate_cancer/epithelial/RENAMED_")
}

In [ ]:
if (renamed == FALSE){
    int@meta.data$seurat_clusters <- int@meta.data$RNA_snn_res.0.5
}
Idents(int) <- "seurat_clusters"
ggsave(
    filename = paste0(wd, "umap_post_integration_clusters.pdf"),
    plot = DimPlot(int, reduction = "umap", group.by = c("sample", "subtype", "seurat_clusters"), ncol = 2, label = TRUE, raster = FALSE),
    width = 16,
    height = 12
)



## GSEA Epithelial subpopulations


In [ ]:
library(singleseqgset)
library(msigdbr)

#### Select Gene Set of interest
# H: hallmark gene sets
# C1: positional gene sets
# C2: curated gene sets

category <- "H" # category from Human MSigDB Collections
h.human <- msigdbr(species="Homo sapiens",category=category)
subcategory <- ""
h.human <- h.human %>% filter(gs_subcat == subcategory)
h.names <- unique(h.human$gs_name)
h.sets <- vector("list",length=length(h.names))
names(h.sets) <- h.names
for (i in names(h.sets)) {
    h.sets[[i]] <- pull(h.human[h.human$gs_name==i,"gene_symbol"])
}
#### Filter out non relevant pathways
# Define the list of pathways to remove
remove_pathways <- c(
  "HALLMARK_BILE_ACID_METABOLISM",         # Liver-specific  
  "HALLMARK_PANCREAS_BETA_CELLS",          # Pancreatic function  
  "HALLMARK_SPERMATOGENESIS",              # Testis-specific  
  "HALLMARK_MYOGENESIS",                   # Muscle development  
  "HALLMARK_ADIPOGENESIS",                 # Fat cell differentiation  
  "HALLMARK_FATTY_ACID_METABOLISM",        # General lipid metabolism  
  "HALLMARK_HEME_METABOLISM",              # Blood-related, not CAF-specific  
  "HALLMARK_CHOLESTEROL_HOMEOSTASIS",      # Unrelated lipid metabolism  
  "HALLMARK_PEROXISOME",                   # General cell detox pathway  
  "HALLMARK_XENOBIOTIC_METABOLISM",        # Detox, not breast cancer-related  
  "HALLMARK_UV_RESPONSE_DN",               # Skin cancer-related  
  "HALLMARK_UV_RESPONSE_UP"                # Skin cancer-related  
)
# Filter the list, keeping only relevant pathways
h.sets <- h.sets[!names(h.sets) %in% remove_pathways]
##### Change names (aesthetic purposes only)
names(h.sets) <- gsub("HALLMARK_", "", names(h.sets))
#### Log Fold change
Idents(int) <- "seurat_clusters"
int$seurat_clusters <- Idents(int)
mtx <- GetAssayData(int, assay = "RNA", layer = "data")
logfc.epi <- logFC(cluster.ids = int@meta.data$seurat_clusters, expr.mat = mtx)
names(logfc.epi)
#### Run GSEA
gse.res <- wmw_gsea(expr.mat=mtx, cluster.cells=logfc.epi[[1]],
                    log.fc.cluster=logfc.epi[[2]],
                    gene.sets=h.sets)
res.stats <- gse.res[["GSEA_statistics"]]
res.pvals <- gse.res[["GSEA_p_values"]]
res.pvals <- apply(res.pvals,2,p.adjust,method="fdr") #Correct for multiple comparisons
res.stats[order(res.stats[,1],decreasing=TRUE)[1:10],] #Top gene sets enriched by z scores
#Plot the z scores with heatmap3
pdf(paste0(wd, "GSEA.pdf"), width = 12, height = 12)
heatmap3(res.stats, margins = c(7, 34))
dev.off()



#### add subclusters back to main object


In [ ]:
if (renamed == TRUE){
    int$renamed_clusters <- paste0("", int$seurat_clusters)
} else {
    int$renamed_clusters <- paste0("Epithelial_", int$seurat_clusters)
}
renamed_clusters <- int$renamed_clusters
names(renamed_clusters) <- Cells(int)

int.all$epi_clusters_integrated <- NA
int.all$epi_clusters_integrated[names(renamed_clusters)] <- renamed_clusters

# Create a new metadata column for Epithelial subtypes
int.all$seuratAnnotation_major_epiInt <- as.character(int.all$seuratAnnotation_major)
int.all$seuratAnnotation_major_epiInt[int.all$seuratAnnotation_major == "Epithelial"] <- int.all$epi_clusters_integrated[int.all$seuratAnnotation_major == "Epithelial"]
int.all$seuratAnnotation_major_epiInt <- factor(int.all$seuratAnnotation_major_epiInt)

ggsave(
    filename = paste0(wd, "Epithelial_all_integratedUMAP.pdf"),
    plot = DimPlot(int.all, reduction = "umap", group.by = c("seuratAnnotation_major_epiInt"), ncol = 1, label = TRUE, raster = FALSE),
    width = 10,
    height = 10
)



### Integrated Spatial mapping 

###### map 


In [ ]:
samples <- unique(int.all$sample)
spls <- list()
for (sample in samples){
    vis <- readRDS(paste0(SPATIAL_RDS, "/prostate_cancer/", sample, "_visHD.rds"))
    vis <- SubsetSTData(vis, expression = resolution == "016um") # use this resolution 

    ## Map w/ semla NNLS
    ident <- "seuratAnnotation_major_epiInt"
    celltype <- "major_integrated_Epithelial"
    DefaultAssay(vis) <- "Spatial"
    DefaultAssay(int.all) <- "RNA"
    vis <- vis |>
    NormalizeData() |>
    FindVariableFeatures(nfeatures = nrow(vis)) |>
    ScaleData()
    int.all <- int.all |>
    FindVariableFeatures(nfeatures = 5000)
    vis <- RunNNLS(vis, singlecell_object = int.all, spatial_assay = DefaultAssay(vis), singlecell_assay = DefaultAssay(int.all), groups = ident, slot = "data", assay_name = celltype)
    DefaultAssay(vis) <- celltype

    spls[[sample]] <- vis
}
mergedVis <- MergeSTData(spls[[1]], spls[-1])
table(mergedVis@meta.data$sample_id)
DefaultAssay(mergedVis) <- celltype

p <- MapFeatures(mergedVis, features = rownames(mergedVis)[grepl("Epithelial", rownames(mergedVis))], colors = viridis::turbo(11), label_by = "sample_id", pt_size = 0.35, override_plot_dims = TRUE, scale = "shared")
ggsave(paste0(wd, "map/", "all_integratedMap_patient_specific.jpg"), p, width = 36, height = 18, dpi = 300, limitsize = FALSE)
# p <- MapFeatures(mergedVis, features = rownames(mergedVis), colors = viridis::turbo(11), label_by = "sample_id", pt_size = 0.4, override_plot_dims = TRUE, scale = "shared")
# ggsave(paste0(paste0(FIGS_ROOT, "/prostate_cancer/total_integration/"), "map/","all_integratedMap.jpg"), p, width = 105, height = 20, dpi = 300, limitsize = FALSE)

In [ ]:
saveRDS(int, paste0(rootObj, "integrated_epithelial.rds"))
saveRDS(int.all, paste0(paste0(SN_RDS, "/prostate_cancer/integrated/"), "integrated_all_epithelial.rds"))



### Epithelial cell state markers


In [ ]:
Idents(int) <- "seurat_clusters"
epi.markers <- FindAllMarkers(int, assay = "RNA")
epi.markers %>%
    group_by(cluster) %>%
    top_n(n = 4, wt = avg_log2FC) -> top

p <- DotPlot(int, features = top$gene) + 
        coord_flip() +
        scale_color_gradientn(colors = c("blue", "white", "darkred")) +
        theme(axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1))
ggsave(paste0(wd, "markers_dotplot.pdf"), p, width = 8, height = 10)



## Rename clusters to putative populations


In [ ]:
int@meta.data$epi_clusters_integratedRenamed <- int@meta.data$seurat_clusters
epi.renames <- c(
)
int@meta.data$epi_clusters_integratedRenamed <- plyr::mapvalues(int@meta.data$epi_clusters_integratedRenamed, from = names(epi.renames), to = epi.renames)
int@meta.data$seurat_clusters <- int@meta.data$epi_clusters_integratedRenamed



################################################################## Ad Hoc Visualizations #####################################################################################################

### Comparison w/ Deconvolution with Prostate Cancer reference


In [ ]:
ref <- readRDS(paste0(EXT_REFS, "/PCa_atlas/PCa_atlas_annotated_Oct23_update.rds"))
int.all <- readRDS(paste0(paste0(SN_RDS, "/prostate_cancer/integrated/"), "integrated_all_epithelial.rds"))



#### Deconvolve by sample 


In [ ]:
samples <- unique(int.all$sample)
spls <- list()
for (sample in samples[2]){
    vis <- readRDS(paste0(SPATIAL_RDS, "/prostate_cancer/", sample, "_visHD.rds"))
    vis <- SubsetSTData(vis, expression = resolution == "016um") # use this resolution 

    ## Map w/ semla NNLS
    identSimp <- "seuratAnnotation_major_epiInt"
    celltypeSimp <- "Simplex_Epithelial"
    identRef <- "celltype_minor_v2"
    celltypeRef <- "Ref_Atlas"

    DefaultAssay(vis) <- "Spatial"
    DefaultAssay(int.all) <- "RNA"
    DefaultAssay(ref) <- "RNA"

    vis <- vis |>
    NormalizeData() |>
    FindVariableFeatures(nfeatures = nrow(vis)) |>
    ScaleData()

    Idents(int.all) <- identSimp
    int.all <- int.all |>
    FindVariableFeatures(nfeatures = 5000)

    Idents(ref) <- identRef
    ref <- ref |>
    FindVariableFeatures(nfeatures = 5000)

    vis <- RunNNLS(vis, singlecell_object = int.all, spatial_assay = DefaultAssay(vis), singlecell_assay = DefaultAssay(int.all), groups = identSimp, slot = "data", assay_name = celltypeSimp)
    DefaultAssay(vis) <- "Spatial"
    vis <- RunNNLS(vis, singlecell_object = ref, spatial_assay = DefaultAssay(vis), singlecell_assay = DefaultAssay(ref), groups = identRef, slot = "data", assay_name = celltypeRef)

    plots <- list()
    for (assay in c(celltypeSimp, celltypeRef)){
        DefaultAssay(vis) <- assay
        if (assay == celltypeSimp){
            features <- rownames(vis)[grepl("Epithelial", rownames(vis))]
        } else {
            features <- c("Basal", "Luminal", "Cycling-Epi-Basal", "Club", "Luminal-like", "Luminal-IFN", "Ciliated", "Cycling-Epi-Club")
        }
        p <- MapFeatures(vis, features = features, colors = viridis::turbo(11), label_by = "sample_id", pt_size = 0.35, override_plot_dims = TRUE, scale = "shared", ncol = length(features))
        plots[[assay]] <- p & scale_fill_viridis_c(limits = c(0,1), option = "H")
    }
    final_plot <- patchwork::wrap_plots(plots, ncol = 1)
    ggsave(paste0(wd, "map/", sample, "_ref_versus_simplex_Epithelial.jpg"), final_plot, height = 10, width = 30, dpi = 300, limitsize = FALSE)
}



##### Ref atlas comparison 


In [ ]:
wd <- paste0(FIGS_ROOT, "/prostate_cancer/total_integration/")
samples <- c("pt10", "pt20")
spls <- list()
for (sample in samples){
    vis <- readRDS(paste0(SPATIAL_RDS, "/prostate_cancer/", sample, "_visHD.rds"))
    vis <- SubsetSTData(vis, expression = resolution == "016um") # use this resolution 

    ## Map w/ semla NNLS
    ident <- "celltype_major_v2"
    celltype <- "ref_atlas_map"
    DefaultAssay(vis) <- "Spatial"
    DefaultAssay(ref) <- "RNA"
    vis <- vis |>
    NormalizeData() |>
    FindVariableFeatures(nfeatures = nrow(vis)) |>
    ScaleData()
    ref <- ref |>
    FindVariableFeatures(nfeatures = 5000)
    vis <- RunNNLS(vis, singlecell_object = ref, spatial_assay = DefaultAssay(vis), singlecell_assay = DefaultAssay(ref), groups = ident, slot = "data", assay_name = celltype)
    DefaultAssay(vis) <- celltype
    spls[[sample]] <- vis
}

# Plot each sample separately
sample_plots <- list()
for (sample in samples) {
    features = unique(gsub("_", "-", ref$celltype_major_v2))
    vis <- spls[[sample]]
    if (sample == "pt10" || sample == "pt20") {
        pt_size <- 0.35
    } else if (sample == "" || sample == "") {
        pt_size <- 1.5
    }
    p <- MapFeatures(vis, features = features, colors = viridis::turbo(11), label_by = "sample_id", pt_size = pt_size, override_plot_dims = TRUE, scale = "shared", ncol = length(features), shape = "tile")
    sample_plots[[sample]] <- p & scale_fill_viridis_c(limits = c(0,1), option = "H")
}
# Merge all sample plots into a final plot
final_plot <- patchwork::wrap_plots(sample_plots, ncol = 1)
ggsave(paste0(wd, "map/", "ref_atlas_HD.jpg"), final_plot, height = 5*length(samples), width = 3 * length(features), dpi = 300, limitsize = FALSE)



## Distribution of patients across labels


In [ ]:
obj <- readRDS(paste0(SN_RDS, "/prostate_cancer/integrated/integrated_all_epithelial.rds"))
wd <- paste0(FIGS_ROOT, "/prostate_cancer/epithelial/")

In [ ]:
obj <- subset(obj, subset = seuratAnnotation_major %in% c("Epithelial"))
# Create a data frame for plotting
plot_data <- obj@meta.data %>%
    dplyr::count(seuratAnnotation_major_epiInt, sample) %>%
    dplyr::group_by(seuratAnnotation_major_epiInt) %>%
    dplyr::mutate(percentage = n / sum(n) * 100)
# rm(obj)
# Plot the stacked histogram
p <- ggplot(plot_data, aes(x = seuratAnnotation_major_epiInt, y = percentage, fill = sample)) +
            geom_bar(stat = "identity") +
            labs(x = "Subpopulations", y = "Percentage", fill = "Patient") +
            theme_minimal() +
            theme(axis.text.x = element_text(angle = 45, hjust = 1))

ggsave(
    filename = paste0(wd, "subcluster_patient_histogram.pdf"),
    plot = p,
    width = 5,
    height = 5
)